# Intrinsic Wavefront Analysis and Plots

**Author:** Aaron Roodman  
**Date Created:** 2026-02-23  
**Last Modified:** 2026-03-17  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Full Array Mode, Zernike, Analysis

## Description

This notebook analyzes the FAM Zernike table created by `intrinsics_mktable.ipynb`.

**Analysis includes:**
1. Load parquet file with Zernike measurements and visit_info companion table
2. Identify FAM blocks (contiguous triplets with visits separated by 3) and print summary
3. Filter out sparse days (< 5 FAM triplet seq_num)
4. Robust per-image linear fit (constant + x,y slopes) to data-model residuals using RLM
5. Plot fit parameters and residual histograms per day_obs
6. Single-image residual maps and histograms for Z4-Z11
7. Create hexbin visualizations: Data, Model, and Data-Model residuals

**Input:** Parquet files created by mktable notebook (in `output/`)  
**Output:** Trio comparison plots (PNG), fit parameter plots, single-image residual maps, FAM block summary

**Based on:** `intrinsics_mktable.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-23 | Aaron Roodman | Initial version |
| 2026-03-15 | Aaron Roodman | Added CCS/OCS coord_sys parameter; added per-image linear fit (constant + x,y slopes) replacing simple mean subtraction; added fit parameter plots vs image for Z4-Z15; updated trio plots to use linear fit subtraction; added rotator angle filter (default -3 to 3 deg) |
| 2026-03-15 | Aaron Roodman | Fixed iZs: infer Zernike indices from data shape instead of hardcoding (table has Z4-Z22 contiguous, 19 terms) |
| 2026-03-15 | Aaron Roodman | Added day_obs_plot parameter for selecting subset of days for plots; fixed k2/k3 units to μm/2deg; added fit residual histograms (log scale, ±1 μm) for Z4-Z15 to study outliers |
| 2026-03-17 | Aaron Roodman | Switched per-image fit from OLS to robust RLM (Huber's T); saves per-donut RLM weights; added single-image residual map+histogram method (plot_single_image_residuals) for Z4-Z11; plots saved to output/ dir with date range in titles |
| 2026-03-17 | Aaron Roodman | Consolidated per-day plotting: loop over all day_obs showing first inline, rest saved to output/; fixed fit_table_plot reference bug in single-image cell; default day_obs range 1023-1231; parquet read from output/ |
| 2026-03-17 | Aaron Roodman | Added wep_ver/dviz_ver params for parquet filenames; load visit_info companion table; FAM block identification (visits separated by 3) with summary table showing day_obs, seq_num range, band, alt, az, rotAngle, Z4 first/last |
| 2026-03-17 | Aaron Roodman | Fit params + histograms into single multi-page PDF per day; replaced hexbin single-image plots with binned_statistic_2d 4x3 grid (Z4-Z15, median, 15 bins, 2/98% color limits); dropped histograms from single-image view; auto-generate MPEG movie of all visits; default day_obs_min=20251020; added band to single-image titles |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load Data](#load)
4. [FAM Block Analysis](#blocks)
5. [Helper Functions](#functions)
6. [Analysis](#analysis)
7. [Linear Fit](#fit)
8. [Fit Parameter Plots & Residual Histograms](#fitplots)
9. [Single-Image Residual Maps](#singleimage)
10. [Visualizations](#viz)

<a id='params'></a>
## Parameters

In [ ]:
from pathlib import Path

# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system: 'CCS' or 'OCS' (must match mktable choice)
coord_sys = 'OCS'

# Parquet file version strings (must match mktable output)
wep_ver = 'wep_v16_8_0'
dviz_ver = 'dviz_v3_5_0'

# Parquet file dates
day_obs_min = 20251020
day_obs_max = 20251231

# Date range string for plot titles
date_range_str = f'{day_obs_min}-{day_obs_max}'

# Rotator angle filter (degrees) — only keep images within this range
rotator_angle_min = -3.0
rotator_angle_max = 3.0

# Minimum unique seq_num per day_obs to keep (filters sparse days)
min_seq_num_per_day = 5

# Zernike indices: inferred from data after loading (see Load Data section)
# iZs and iZidx are set dynamically below

# Plotting parameters
plo_default = 4.0   # Low percentile for colorbar
phi_default = 96.0  # High percentile for colorbar
output_dir = 'output'

# Ensure output directory exists
Path(output_dir).mkdir(parents=True, exist_ok=True)

<a id='setup'></a>
## Setup & Imports

In [ ]:
# Standard imports
from matplotlib import pyplot as plt
from matplotlib import lines
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import pandas as pd
from scipy.stats import binned_statistic_2d, binned_statistic
from pathlib import Path
import statsmodels.api as sm
import subprocess

# Astropy
from astropy.table import Table, QTable

# Set plot defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Pandas display options
pd.options.display.max_columns = None
pd.options.display.width = None

<a id='load'></a>
## Load Data

In [ ]:
# Load the main donut parquet file and visit_info companion table
parquet_file = f'{output_dir}/fam_zernikes_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}.parquet'
visit_info_file = f'{output_dir}/fam_zernikes_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}_visits.parquet'

if not Path(parquet_file).exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_file}\n"
                           f"Run intrinsics_mktable.ipynb first to create it.")

print(f"Loading data from: {parquet_file}")
aosTable = QTable.read(parquet_file)
print(f"Loaded {len(aosTable)} rows, {len(aosTable.columns)} columns")
print(f"Coordinate system: {coord_sys}")

# Load visit_info table
if Path(visit_info_file).exists():
    visit_info = QTable.read(visit_info_file)
    print(f"\nLoaded visit_info from: {visit_info_file}")
    print(f"  {len(visit_info)} visits, columns: {visit_info.colnames}")
else:
    visit_info = None
    print(f"\nWarning: visit_info file not found: {visit_info_file}")
    print("  FAM block analysis will be skipped.")

In [3]:
# Display table summary
print("\nTable columns:")
print(sorted(aosTable.columns))

# Count by day_obs
print("\nMeasurements per day_obs:")
day_counts = pd.DataFrame({'day_obs': aosTable['day_obs']}).value_counts().sort_index()
print(day_counts)


Table columns:
['centroid_x', 'centroid_x_extra', 'centroid_x_intra', 'centroid_y', 'centroid_y_extra', 'centroid_y_intra', 'coord_dec', 'coord_dec_extra', 'coord_dec_intra', 'coord_ra', 'coord_ra_extra', 'coord_ra_intra', 'day_obs', 'detector', 'extra_fpx', 'extra_fpy', 'intra_fpx', 'intra_fpy', 'matched_intra_extra', 'seq_num', 'th_N', 'th_N_extra', 'th_N_intra', 'th_W', 'th_W_extra', 'th_W_intra', 'thx_CCS', 'thx_CCS_extra', 'thx_CCS_intra', 'thx_OCS', 'thx_OCS_extra', 'thx_OCS_intra', 'thy_CCS', 'thy_CCS_extra', 'thy_CCS_intra', 'thy_OCS', 'thy_OCS_extra', 'thy_OCS_intra', 'used', 'zk_CCS', 'zk_CCS_mean', 'zk_NW', 'zk_OCS', 'zk_intrinsic', 'zk_residual']

Measurements per day_obs:
day_obs 
20250825    120458
20250826     94567
20250907      1586
20250909      7029
20250912     11256
20250913     52244
20251023     44375
20251024      9162
20251026     79514
20251027     20498
20251028      7374
Name: count, dtype: int64


In [4]:
# Display sample rows with formatted floats
# Select scalar columns only
scalar_cols = []
for col in aosTable.columns:
    if not hasattr(aosTable[col][0], '__len__') or isinstance(aosTable[col][0], str):
        scalar_cols.append(col)

df_display = aosTable[scalar_cols].to_pandas()
pd.options.display.float_format = '{:.2f}'.format

print("\nSample rows (first 5):")
print(df_display.head())

pd.reset_option('display.float_format')


Sample rows (first 5):
  detector  used  coord_ra  coord_dec  centroid_x  centroid_y  thx_CCS  \
0  R01_S00  True      4.72      -0.38     3773.04     3752.49    -0.03   
1  R01_S01  True      4.72      -0.38     1991.94     3338.47    -0.03   
2  R01_S01  True      4.72      -0.38     2997.63     2987.60    -0.03   
3  R01_S01  True      4.72      -0.38     2273.00     2673.47    -0.03   
4  R01_S01  True      4.72      -0.38      988.99     2905.48    -0.03   

   thy_CCS  thx_OCS  thy_OCS  th_N  th_W  coord_ra_intra  coord_ra_extra  \
0    -0.01    -0.03    -0.01  0.03 -0.02            4.72            4.72   
1    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
2    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
3    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
4    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   

   coord_dec_intra  coord_dec_extra  centroid_x_intra  centroid_x_extra  \

<a id='blocks'></a>
## FAM Block Analysis

Identify contiguous blocks of Full Array Mode (FAM) triplets from the visit_info table.
FAM triplets (intra, in-focus, extra-focal) have visits separated by increments of 3.

In [ ]:
# Identify FAM blocks from visit_info
# FAM triplets (intra, in-focus, extra) have visits separated by increments of 3.
# A FAM block is a contiguous group of visits where consecutive visits differ by exactly 3.

if visit_info is not None:
    visits_sorted = visit_info.copy()
    visits_sorted.sort('visit')
    visit_arr = np.array(visits_sorted['visit'])
    
    # Get mean Z4 (first element of zk_OCS) per visit from the main table
    day_obs_main = np.array(aosTable['day_obs'])
    seq_num_main = np.array(aosTable['seq_num'])
    zk_data_all = np.stack(aosTable[f'zk_{coord_sys}'])
    
    # Compute mean Z4 per visit
    visit_z4_mean = {}
    for row in visits_sorted:
        dobs = row['day_obs']
        snum = row['seq_num']
        mask = (day_obs_main == dobs) & (seq_num_main == snum)
        if np.sum(mask) > 0:
            visit_z4_mean[(dobs, snum)] = np.mean(zk_data_all[mask, 0])
        else:
            visit_z4_mean[(dobs, snum)] = np.nan
    
    # Find block boundaries: consecutive visits differ by 3
    diffs = np.diff(visit_arr)
    block_breaks = np.where(diffs != 3)[0] + 1
    block_starts = np.concatenate([[0], block_breaks])
    block_ends = np.concatenate([block_breaks, [len(visit_arr)]])
    
    print(f"Found {len(block_starts)} FAM blocks from {len(visit_arr)} visits\n")
    
    # Build summary table
    block_summary = []
    for b_idx, (bs, be) in enumerate(zip(block_starts, block_ends)):
        block_rows = visits_sorted[bs:be]
        n_visits = len(block_rows)
        dobs = int(block_rows['day_obs'][0])
        seq_min = int(np.min(block_rows['seq_num']))
        seq_max = int(np.max(block_rows['seq_num']))
        band = str(block_rows['band'][0])
        alt_mean = float(np.mean(block_rows['alt']))
        az_mean = float(np.mean(block_rows['az']))
        rot_mean = float(np.mean(block_rows['rotAngle']))
        
        # Z4 of first and last image in block
        first_key = (int(block_rows['day_obs'][0]), int(block_rows['seq_num'][0]))
        last_key = (int(block_rows['day_obs'][-1]), int(block_rows['seq_num'][-1]))
        z4_first = visit_z4_mean.get(first_key, np.nan)
        z4_last = visit_z4_mean.get(last_key, np.nan)
        
        block_summary.append({
            'block': b_idx,
            'day_obs': dobs,
            'seq_min': seq_min,
            'seq_max': seq_max,
            'n_visits': n_visits,
            'band': band,
            'alt': alt_mean,
            'az': az_mean,
            'rotAngle': rot_mean,
            'Z4_first': z4_first,
            'Z4_last': z4_last,
        })
    
    block_df = pd.DataFrame(block_summary)
    pd.options.display.float_format = '{:.2f}'.format
    print("FAM Block Summary:")
    print(block_df.to_string(index=False))
    pd.reset_option('display.float_format')
else:
    print("visit_info not available — skipping FAM block analysis")

In [ ]:
# Filter out day_obs with fewer than min_seq_num_per_day unique seq_num
# These are days with too few FAM triplets for reliable analysis
day_obs_all = np.array(aosTable['day_obs'])
seq_num_all = np.array(aosTable['seq_num'])

unique_days = sorted(set(day_obs_all.tolist()))
print(f"Before filtering: {len(unique_days)} days, {len(aosTable)} donuts")
print(f"Unique seq_num per day_obs (minimum required: {min_seq_num_per_day}):")

sparse_days = []
for d in unique_days:
    n_unique_seq = len(set(seq_num_all[day_obs_all == d].tolist()))
    n_donuts_day = np.sum(day_obs_all == d)
    status = "REMOVED" if n_unique_seq < min_seq_num_per_day else "kept"
    print(f"  {d}: {n_unique_seq} seq_num, {n_donuts_day} donuts  [{status}]")
    if n_unique_seq < min_seq_num_per_day:
        sparse_days.append(d)

if sparse_days:
    sparse_mask = ~np.isin(day_obs_all, sparse_days)
    n_before = len(aosTable)
    aosTable = aosTable[sparse_mask]
    n_after = len(aosTable)
    print(f"\nRemoved {len(sparse_days)} day_obs with < {min_seq_num_per_day} unique seq_num: {sparse_days}")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
else:
    print(f"\nAll day_obs have >= {min_seq_num_per_day} unique seq_num")

# Summary of remaining data
remaining_days = sorted(set(np.array(aosTable['day_obs']).tolist()))
n_images = len(set(zip(np.array(aosTable['day_obs']).tolist(), np.array(aosTable['seq_num']).tolist())))
print(f"\nRemaining: {len(remaining_days)} days, {n_images} images, {len(aosTable)} donuts")

In [ ]:
# Filter by rotator angle
if 'rotator_angle' in aosTable.columns:
    rot_angles = np.array(aosTable['rotator_angle'])
    rot_mask = (rot_angles >= rotator_angle_min) & (rot_angles <= rotator_angle_max)
    n_before = len(aosTable)
    aosTable = aosTable[rot_mask]
    n_after = len(aosTable)
    n_images_before = len(set(zip(np.array(aosTable['day_obs']), np.array(aosTable['seq_num']))))
    print(f"Rotator angle filter [{rotator_angle_min}, {rotator_angle_max}] deg:")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
    print(f"  Remaining images: {n_images_before}")
else:
    print("Warning: 'rotator_angle' column not found in table.")
    print("  Rotator angle filter NOT applied.")
    print("  Re-run intrinsics_mktable.ipynb to include rotator angles.")

<a id='functions'></a>
## Helper Functions

In [ ]:
def get_zernike(table, column_name, iZ):
    """Extract a single Zernike term from an array column.
    
    Parameters
    ----------
    table : QTable
        Table with Zernike array columns
    column_name : str
        Column name (e.g. 'zk_OCS', 'zk_intrinsic', 'zk_residual')
    iZ : int
        Zernike index (4-28, excluding 20, 21)
    
    Returns
    -------
    array : ndarray
        Zernike values
    """
    if iZ not in iZidx:
        raise ValueError(f"Zernike Z{iZ} not in table. Available: {iZs}")
    
    zk_array = np.stack(table[column_name])
    return zk_array[:, iZidx[iZ]]


def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Add a vertical color bar to an image plot."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1./aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


def plot_fit_params_and_residuals(fit_table_day, aosTable_matched, plot_mask_day,
                                  day_obs_list, iZs_fit_plot=None, iZs_hist=None,
                                  output_dir='.', show=True):
    """Plot k1/k2/k3 fit parameters and residual histograms into a single PDF.
    
    Produces a multi-page PDF with one page each for k1, k2, k3 (3x4 grid of
    Zernikes) and one page for residual histograms (3x4 grid).
    
    Parameters
    ----------
    fit_table_day : QTable
        Fit parameters table filtered to selected days.
    aosTable_matched : QTable
        Full matched donut table (with zk_fit column).
    plot_mask_day : ndarray (bool)
        Mask into aosTable_matched for the selected days.
    day_obs_list : list of int
        Day_obs values being plotted (used in titles).
    iZs_fit_plot : list of int, optional
        Zernike indices for fit param plots (default [4..15]).
    iZs_hist : list of int, optional
        Zernike indices for residual histograms (default [4..15]).
    output_dir : str
        Directory for saved figures.
    show : bool
        If True, display plots inline. If False, save and close.
    """
    if iZs_fit_plot is None:
        iZs_fit_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    if iZs_hist is None:
        iZs_hist = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    
    # Build title label from day_obs_list
    if len(day_obs_list) == 1:
        day_label = str(day_obs_list[0])
    elif len(day_obs_list) <= 4:
        day_label = ', '.join(str(d) for d in day_obs_list)
    else:
        day_label = f'{day_obs_list[0]}...{day_obs_list[-1]} ({len(day_obs_list)} days)'
    
    # File suffix for output names
    if len(day_obs_list) == 1:
        file_suffix = str(day_obs_list[0])
    else:
        file_suffix = 'all'
    
    pdf_path = f'{output_dir}/fit_params_resid_{file_suffix}.pdf'
    
    with PdfPages(pdf_path) as pdf:
        # --- Fit parameter plots: k1, k2, k3 vs image index ---
        param_names = ['k1', 'k2', 'k3']
        param_labels = ['k1 (offset)', 'k2 (x slope)', 'k3 (y slope)']
        param_units = ['μm', 'μm/2deg', 'μm/2deg']
        image_idx = np.arange(len(fit_table_day))
        
        for p_idx, (pname, plabel, punit) in enumerate(zip(param_names, param_labels, param_units)):
            fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
            fig.suptitle(f'Fit Parameter {plabel} vs Image (day_obs: {day_label})', fontsize=14)
            
            for ax_idx, iZ in enumerate(iZs_fit_plot):
                ax = axes[ax_idx // 4, ax_idx % 4]
                vals = np.array(fit_table_day[f'z{iZ}_{pname}'])
                ax.plot(image_idx, vals, 'o-', markersize=3)
                ax.set_title(f'Z{iZ}')
                ax.set_ylabel(punit)
                ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
                if ax_idx // 4 == 2:
                    ax.set_xlabel('Image index')
            
            plt.tight_layout()
            pdf.savefig(fig, dpi=150, bbox_inches='tight')
            if show:
                plt.show()
            else:
                plt.close(fig)
        
        # --- Fit residual histograms ---
        zk_data_plot = np.stack(aosTable_matched[f'zk_{coord_sys}'])[plot_mask_day]
        zk_model_plot = np.stack(aosTable_matched['zk_intrinsic'])[plot_mask_day]
        zk_fit_plot = np.stack(aosTable_matched['zk_fit'])[plot_mask_day]
        
        fig, axes = plt.subplots(3, 4, figsize=(18, 10))
        fig.suptitle(f'Fit Residuals: data - model - linear fit (day_obs: {day_label})', fontsize=14)
        
        hist_range = (-1.0, 1.0)
        n_bins = 100
        
        for ax_idx, iZ in enumerate(iZs_hist):
            ax = axes[ax_idx // 4, ax_idx % 4]
            j = iZidx[iZ]
            
            resid = zk_data_plot[:, j] - zk_model_plot[:, j] * 1e6 - zk_fit_plot[:, j]
            
            n_total = len(resid)
            n_in = np.sum((resid >= hist_range[0]) & (resid <= hist_range[1]))
            n_out = n_total - n_in
            
            ax.hist(resid, bins=n_bins, range=hist_range, log=True,
                    edgecolor='black', linewidth=0.3, alpha=0.7)
            ax.set_title(f'Z{iZ}')
            ax.set_xlabel('μm')
            ax.set_ylabel('Count')
            ax.set_xlim(hist_range)
            
            std_val = np.std(resid)
            ax.text(0.97, 0.95, f'σ={std_val:.3f} μm\n{n_out}/{n_total} outside',
                    transform=ax.transAxes, ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        pdf.savefig(fig, dpi=150, bbox_inches='tight')
        if show:
            plt.show()
        else:
            plt.close(fig)
    
    print(f"Saved: {pdf_path}")


def plot_single_image_residual_grid(aosTable_matched, day_obs, seq_num,
                                     band='', iZs_plot=None,
                                     n_steps=16, statistic='median',
                                     plo=2.0, phi=98.0,
                                     fpradius=1.8,
                                     output_dir='.', show=True):
    """Plot a 4x3 grid of binned residual maps (Z4-Z15) for a single image.
    
    Uses binned_statistic_2d with a fixed grid for consistent appearance
    across images.  No histograms — just the focal-plane maps.
    
    Parameters
    ----------
    aosTable_matched : QTable
        Matched donut table (must include 'zk_fit' column).
    day_obs : int
        Day of observation.
    seq_num : int
        Sequence number within the day.
    band : str, optional
        Filter band name (for title annotation).
    iZs_plot : list of int, optional
        Zernike indices to plot (default [4..15], arranged 4x3).
    n_steps : int, optional
        Number of bin edges along each axis (default 16, giving 15 bins).
    statistic : str, optional
        Statistic for binned_statistic_2d (default 'median').
    plo, phi : float, optional
        Percentile limits for color scale (default 2, 98).
    fpradius : float, optional
        Focal plane radius in degrees (default 1.8).
    output_dir : str
        Directory for saved figure.
    show : bool
        If True, display inline.  If False, save and close (for movie).
    
    Returns
    -------
    output_file : str or None
        Path to saved PNG, or None if no donuts found.
    """
    if iZs_plot is None:
        iZs_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    
    dobs_arr = np.array(aosTable_matched['day_obs'])
    snum_arr = np.array(aosTable_matched['seq_num'])
    img_mask = (dobs_arr == day_obs) & (snum_arr == seq_num)
    n_donuts = np.sum(img_mask)
    
    if n_donuts == 0:
        return None
    
    # Field angles in degrees (x,y reversed for Rubin visualization standard)
    xval = np.rad2deg(np.array(aosTable_matched[f'thy_{coord_sys}_extra'])[img_mask])
    yval = np.rad2deg(np.array(aosTable_matched[f'thx_{coord_sys}_extra'])[img_mask])
    
    zk_data_img = np.stack(aosTable_matched[f'zk_{coord_sys}'])[img_mask]
    zk_model_img = np.stack(aosTable_matched['zk_intrinsic'])[img_mask]
    zk_fit_img = np.stack(aosTable_matched['zk_fit'])[img_mask]
    
    # Fixed binning grid
    bins_edge = np.linspace(-fpradius, fpradius, n_steps)
    
    n_rows, n_cols = 3, 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 12))
    
    band_str = f'  band={band}' if band else ''
    fig.suptitle(f'Single-Image Residuals: day_obs={day_obs}  seq_num={seq_num}{band_str}'
                 f'  ({n_donuts} donuts)', fontsize=14)
    
    for ax_idx, iZ in enumerate(iZs_plot):
        ax = axes[ax_idx // n_cols, ax_idx % n_cols]
        j = iZidx[iZ]
        resid = zk_data_img[:, j] - zk_model_img[:, j] * 1e6 - zk_fit_img[:, j]
        
        # Binned statistic on fixed grid
        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, resid,
            statistic=statistic, bins=[bins_edge, bins_edge])
        
        # Percentile-based color limits from the finite values
        finite_vals = stat_val[np.isfinite(stat_val)]
        if len(finite_vals) > 0:
            vmin, vmax = np.percentile(finite_vals, [plo, phi])
        else:
            vmin, vmax = -1.0, 1.0
        
        im = ax.imshow(
            stat_val.T, origin='lower',
            extent=[-fpradius, fpradius, -fpradius, fpradius],
            cmap='RdBu_r', interpolation='none', aspect='equal',
            vmin=vmin, vmax=vmax)
        fig.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(f'Z{iZ}', fontsize=11)
        ax.set_xlim(-fpradius, fpradius)
        ax.set_ylim(-fpradius, fpradius)
        if ax_idx % n_cols == 0:
            ax.set_ylabel(f'thx_{coord_sys} [deg]')
        if ax_idx // n_cols == n_rows - 1:
            ax.set_xlabel(f'thy_{coord_sys} [deg]')
    
    plt.tight_layout()
    output_file = f'{output_dir}/single_image_resid_{day_obs}_{seq_num}.jpg'
    fig.savefig(output_file, dpi=120, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return output_file


def plot_zernike_trio(aosTable_matched, iZ, plo=4.0, phi=96.0,
                      output_dir='.', date_range_str=''):
    """Create trio of plots for a Zernike term: Data, Model, Data-Model.
    
    Data is shown with per-image linear fit subtracted (constant + x,y slopes).
    
    Parameters
    ----------
    aosTable_matched : QTable
        Table with matched intra/extra donuts (must include 'zk_fit' column)
    iZ : int
        Zernike index (e.g., 4 for defocus)
    plo, phi : float
        Percentile limits for colorbar
    output_dir : str
        Directory to save figures
    date_range_str : str
        Date range string for plot title
    """
    nsteps = 18 * 4 + 1
    fpradius = 1.8
    xbins = np.linspace(-fpradius, fpradius, nsteps)
    ybins = np.linspace(-fpradius, fpradius, nsteps)
    
    xval = np.rad2deg(aosTable_matched[f'thy_{coord_sys}_extra'])
    yval = np.rad2deg(aosTable_matched[f'thx_{coord_sys}_extra'])
    
    zk_data_all = np.stack(aosTable_matched[f'zk_{coord_sys}'])
    zk_fit_all = np.stack(aosTable_matched['zk_fit'])
    zk_model_all = np.stack(aosTable_matched['zk_intrinsic'])
    
    zval_data = zk_data_all[:, iZidx[iZ]] - zk_fit_all[:, iZidx[iZ]]
    zval_model = zk_model_all[:, iZidx[iZ]] * 1e6
    zval_residual = zval_data - zval_model
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    mean_val_data, _, _, _ = binned_statistic_2d(
        xval, yval, zval_data, statistic='mean', bins=[xbins, ybins])
    vmin_data, vmax_data = np.nanpercentile(zval_data, [plo, phi])
    im0 = axes[0].imshow(
        mean_val_data.T, origin='lower',
        extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
        cmap='viridis', interpolation='none', aspect='equal',
        vmin=vmin_data, vmax=vmax_data)
    add_colorbar(im0, label='μm')
    axes[0].set_xlabel(f'thy_{coord_sys} [deg]')
    axes[0].set_ylabel(f'thx_{coord_sys} [deg]')
    axes[0].set_title(f'Z{iZ} Data (linear fit per visit subtracted)')
    axes[0].set_aspect('equal')
    
    mean_val_model, _, _, _ = binned_statistic_2d(
        xval, yval, zval_model, statistic='mean', bins=[xbins, ybins])
    vmin_model, vmax_model = np.nanpercentile(zval_model, [plo, phi])
    im1 = axes[1].imshow(
        mean_val_model.T, origin='lower',
        extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
        cmap='viridis', interpolation='none', aspect='equal',
        vmin=vmin_model, vmax=vmax_model)
    add_colorbar(im1, label='μm')
    axes[1].set_xlabel(f'thy_{coord_sys} [deg]')
    axes[1].set_ylabel(f'thx_{coord_sys} [deg]')
    axes[1].set_title(f'Z{iZ} Model Intrinsic')
    axes[1].set_aspect('equal')
    
    mean_val_residual, _, _, _ = binned_statistic_2d(
        xval, yval, zval_residual, statistic='mean', bins=[xbins, ybins])
    vmin_res, vmax_res = np.nanpercentile(zval_residual, [plo, phi])
    im2 = axes[2].imshow(
        mean_val_residual.T, origin='lower',
        extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
        cmap='RdBu_r', interpolation='none', aspect='equal',
        vmin=vmin_res, vmax=vmax_res)
    add_colorbar(im2, label='μm')
    axes[2].set_xlabel(f'thy_{coord_sys} [deg]')
    axes[2].set_ylabel(f'thx_{coord_sys} [deg]')
    axes[2].set_title(f'Z{iZ} Data - Model')
    axes[2].set_aspect('equal')
    
    title = f'Z{iZ} Comparison: Data vs Model (matched In/Ex donuts)'
    if date_range_str:
        title += f' ({date_range_str})'
    fig.suptitle(title, fontsize=14, y=0.98)
    
    plt.tight_layout()
    
    output_file = f'{output_dir}/Z{iZ}_trio_comparison.png'
    fig.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f"Saved: {output_file}")
    
    plt.show()
    
    print(f"\nZ{iZ} Statistics (μm):")
    print(f"  Data (fit-sub):   mean={np.nanmean(zval_data):7.2f}, std={np.nanstd(zval_data):7.2f}")
    print(f"  Model:            mean={np.nanmean(zval_model):7.2f}, std={np.nanstd(zval_model):7.2f}")
    print(f"  Residual:         mean={np.nanmean(zval_residual):7.2f}, std={np.nanstd(zval_residual):7.2f}")
    print("="*60)


print("Helper functions loaded: get_zernike(), add_colorbar(), plot_fit_params_and_residuals(),")
print("  plot_single_image_residual_grid(), plot_zernike_trio()")
print(f"\nField angle columns: thx_{coord_sys}, thy_{coord_sys} (radians)")
print(f"Zernike data column: zk_{coord_sys}")

In [ ]:
# Extract Zernike arrays and infer Zernike indices from data shape
zk_data = np.stack(aosTable[f'zk_{coord_sys}'])
zk_model = np.stack(aosTable['zk_intrinsic'])

nZk = zk_data.shape[1]

# Infer Zernike indices (must match infer_zernike_indices in mktable)
if nZk == 19:
    iZs = list(range(4, 23))   # Z4 through Z22 (contiguous)
else:
    iZs = list(range(4, 4 + nZk))
    print(f"Warning: Unexpected number of Zernike terms ({nZk}), assuming Z4-Z{3 + nZk}")

iZidx = {iZ: i for i, iZ in enumerate(iZs)}

print(f"Extracted Zernike arrays:")
print(f"  Shape: {zk_data.shape} (rows x Zernikes)")
print(f"  Zernike indices: {iZs}")
print(f"  Coordinate system: {coord_sys}")

<a id='analysis'></a>
## Analysis

Filter for matched intra/extra donuts, then fit per-image linear model and create trio plots.

In [8]:
# Filter for matched intra/extra donuts
matched_mask = aosTable['matched_intra_extra']
print(f"Total donuts: {len(aosTable)}")
print(f"Matched intra/extra donuts: {np.sum(matched_mask)}")
print(f"Fraction matched: {np.sum(matched_mask)/len(aosTable):.3f}")

# Apply filter
aosTable_matched = aosTable[matched_mask]

Total donuts: 448063
Matched intra/extra donuts: 323608
Fraction matched: 0.722


<a id='fit'></a>
## Linear Fit per Image

For each image and Zernike term, fit: `zJ_data = zJ_model + k1 + k2 * Zfocal_k2 + k3 * Zfocal_k3`  
where `Zfocal_k2 = 2.0 * thx` and `Zfocal_k3 = 2.0 * thy`.

In [ ]:
# Compute robust linear fit per image: (data - model) = k1 + k2 * 2*thx + k3 * 2*thy
# Uses statsmodels RLM (Robust Linear Model) with Huber's T norm to downweight outliers.
# Field angles converted to degrees so slope units are μm/2deg.
thx_col = f'thx_{coord_sys}'
thy_col = f'thy_{coord_sys}'

# Get arrays from matched table (convert radians to degrees)
day_obs_arr = np.array(aosTable_matched['day_obs'])
seq_num_arr = np.array(aosTable_matched['seq_num'])
thx_arr = np.rad2deg(np.array(aosTable_matched[thx_col]))
thy_arr = np.rad2deg(np.array(aosTable_matched[thy_col]))
zk_data_matched = np.stack(aosTable_matched[f'zk_{coord_sys}'])
zk_model_matched = np.stack(aosTable_matched['zk_intrinsic'])

# Identify unique images
images = sorted(set(zip(day_obs_arr.tolist(), seq_num_arr.tolist())))
n_donuts = len(aosTable_matched)
n_zernikes = len(iZs)

# Storage for per-donut fitted values, weights, and per-image fit parameters
zk_fit = np.zeros((n_donuts, n_zernikes))
zk_weights = np.ones((n_donuts, n_zernikes))
fit_params_list = []

for img_idx, (dobs, snum) in enumerate(images):
    mask = (day_obs_arr == dobs) & (seq_num_arr == snum)
    n_pts = np.sum(mask)
    
    Zfocal_k2 = 2.0 * thx_arr[mask]
    Zfocal_k3 = 2.0 * thy_arr[mask]
    
    # Design matrix: [1, Zfocal_k2, Zfocal_k3]
    A = np.column_stack([np.ones(n_pts), Zfocal_k2, Zfocal_k3])
    
    img_params = {'day_obs': dobs, 'seq_num': snum, 'image_idx': img_idx, 'n_donuts': int(n_pts)}
    
    for j_idx, iZ in enumerate(iZs):
        # Data - model residual (model converted from meters to microns)
        resid = zk_data_matched[mask, j_idx] - zk_model_matched[mask, j_idx] * 1e6
        
        # Robust fit using Huber's T norm (downweights outliers)
        try:
            rlm_model = sm.RLM(resid, A, M=sm.robust.norms.HuberT())
            rlm_results = rlm_model.fit()
            k1, k2, k3 = rlm_results.params
            weights = rlm_results.weights
        except Exception:
            # Fallback to OLS if RLM fails (e.g. too few points)
            coeffs, _, _, _ = np.linalg.lstsq(A, resid, rcond=None)
            k1, k2, k3 = coeffs
            weights = np.ones(n_pts)
        
        img_params[f'z{iZ}_k1'] = k1
        img_params[f'z{iZ}_k2'] = k2
        img_params[f'z{iZ}_k3'] = k3
        
        # Compute fitted value for each donut in this image
        zk_fit[mask, j_idx] = k1 + k2 * Zfocal_k2 + k3 * Zfocal_k3
        zk_weights[mask, j_idx] = weights
    
    fit_params_list.append(img_params)

# Add fit values and weights to matched table
aosTable_matched['zk_fit'] = list(zk_fit)
aosTable_matched['zk_rlm_weights'] = list(zk_weights)

# Create fit parameters table
fit_table = QTable(fit_params_list)
print(f"Robust (RLM) linear fit computed for {len(images)} images, {n_donuts} matched donuts")
print(f"  Field angles converted to degrees; slopes in μm/2deg")
print(f"\nFit parameters table: {len(fit_table.columns)} columns")
print(f"Sample (Z4): k1 range = [{fit_table['z4_k1'].min():.3f}, {fit_table['z4_k1'].max():.3f}] μm")

<a id='fitplots'></a>
## Fit Parameter Plots and Residual Histograms

Loop over each day_obs: show the first day inline, save all to output/.

In [ ]:
# Get list of day_obs values remaining after filtering
all_day_obs = sorted(set(np.array(aosTable_matched['day_obs']).tolist()))
print(f"Day_obs values for plots: {all_day_obs}")

# Build arrays needed for masking
day_obs_matched = np.array(aosTable_matched['day_obs'])
fit_day_obs = np.array(fit_table['day_obs'])

In [ ]:
# Plot fit parameters and residual histograms for each day_obs
# First day is shown inline; all days saved to output/
for i, dobs in enumerate(all_day_obs):
    day_mask_fit = fit_day_obs == dobs
    day_mask_matched = day_obs_matched == dobs
    fit_table_day = fit_table[day_mask_fit]
    
    show_inline = (i == 0)
    plot_fit_params_and_residuals(
        fit_table_day, aosTable_matched, day_mask_matched,
        day_obs_list=[dobs],
        output_dir=output_dir, show=show_inline
    )

# Also make combined all-days plots
plot_fit_params_and_residuals(
    fit_table, aosTable_matched, np.ones(len(aosTable_matched), dtype=bool),
    day_obs_list=all_day_obs,
    output_dir=output_dir, show=True
)

<a id='singleimage'></a>
## Single-Image Residual Maps

For each visit, plot a 4x3 grid of binned residual maps (Z4–Z15) using `binned_statistic_2d`.
Show the first visit inline; all visits are saved as JPEGs and combined into an MPEG movie.

In [ ]:
# Generate single-image residual maps for all visits
# First visit shown inline, rest saved to output/ for movie

# Build a lookup for band from visit_info (if available)
band_lookup = {}
if visit_info is not None:
    for row in visit_info:
        band_lookup[(int(row['day_obs']), int(row['seq_num']))] = str(row['band'])

# Get sorted list of all (day_obs, seq_num) in matched table
all_images = sorted(set(zip(
    np.array(aosTable_matched['day_obs']).tolist(),
    np.array(aosTable_matched['seq_num']).tolist()
)))
print(f"Generating residual maps for {len(all_images)} visits...")

frame_files = []
for i, (dobs, snum) in enumerate(all_images):
    band = band_lookup.get((dobs, snum), '')
    show_inline = (i == 0)
    outfile = plot_single_image_residual_grid(
        aosTable_matched, dobs, snum,
        band=band, n_steps=16, statistic='median',
        plo=2.0, phi=98.0,
        output_dir=output_dir, show=show_inline
    )
    if outfile is not None:
        frame_files.append(outfile)
    if (i + 1) % 50 == 0:
        print(f"  ...processed {i + 1}/{len(all_images)} visits")

print(f"Generated {len(frame_files)} residual map frames")

# Combine into MPEG movie using ffmpeg
if len(frame_files) > 1:
    # Write file list for ffmpeg concat demuxer (filenames only, since cwd=output_dir)
    list_file = f'{output_dir}/frame_list.txt'
    with open(list_file, 'w') as f:
        for fpath in frame_files:
            f.write(f"file '{Path(fpath).name}'\n")
            f.write("duration 0.5\n")  # 0.5 seconds per frame = 2 fps
    
    movie_file = f'{output_dir}/single_image_residuals.mp4'
    try:
        # Run ffmpeg with cwd=output_dir, so use just filenames for -i
        result = subprocess.run(
            ['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
             '-i', 'frame_list.txt',
             '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
             '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
             '-r', '2',
             'single_image_residuals.mp4'],
            capture_output=True, text=True, cwd=output_dir
        )
        if result.returncode == 0:
            print(f"Saved movie: {movie_file}")
        else:
            print(f"ffmpeg failed (return code {result.returncode}):")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    except FileNotFoundError:
        print("ffmpeg not found — movie not created. Install ffmpeg to generate MPEG.")
else:
    print("Not enough frames for a movie.")

<a id='viz'></a>
## Visualizations

Create trio plots for each Zernike term: Data (linear fit subtracted), Model, and Data-Model residuals.

In [ ]:
# Build day_obs label for trio plot titles
if len(all_day_obs) <= 4:
    day_obs_plot_str = 'day_obs: ' + ', '.join(str(d) for d in all_day_obs)
else:
    day_obs_plot_str = f'day_obs: {all_day_obs[0]}...{all_day_obs[-1]} ({len(all_day_obs)} days)'

# Select matched donuts for all remaining days (already filtered)
aosTable_plot = aosTable_matched
print(f"Donuts for trio plots: {len(aosTable_plot)} ({day_obs_plot_str})")

# Example: Plot Z4 (defocus)
plot_zernike_trio(aosTable_plot, iZ=4, plo=plo_default, phi=phi_default,
                  output_dir=output_dir, date_range_str=day_obs_plot_str)

In [ ]:
# Plot Zernike terms Z4-Z11
iZs_to_plot = [4, 5, 6, 7, 8, 9, 10, 11]

for iZ in iZs_to_plot:
    plot_zernike_trio(aosTable_plot, iZ=iZ, plo=plo_default, phi=phi_default,
                      output_dir=output_dir, date_range_str=day_obs_plot_str)

In [ ]:
# Plot Zernike terms Z12-Z22
iZs_to_plot = [iZ for iZ in iZs if 12 <= iZ <= 22]

for iZ in iZs_to_plot:
    plot_zernike_trio(aosTable_plot, iZ=iZ, plo=plo_default, phi=phi_default,
                      output_dir=output_dir, date_range_str=day_obs_plot_str)